In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import timm
from tqdm.notebook import tqdm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}. Peringatan: Model Raksasa sedang dipanggil!")

Menggunakan device: cuda:0. Peringatan: Model Raksasa sedang dipanggil!


In [ ]:
# --- 1. HYPERPARAMETER  ---
BATCH_SIZE = 8  
EPOCHS = 8      
LEARNING_RATE = 5e-5 
IMG_SIZE = 224

# Kita gunakan folder ULTIMATE yang sudah berisi kunci jawaban temanmu
data_dir = '../Data/train_ultimate/' 

In [ ]:
# --- 2. AUGMENTASI STANDAR (NO BLUR/NOISE) ---
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=val_transform)
class_names = full_dataset.classes

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

train_dataset.dataset.transform = train_transform
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Dataset Ultimate siap! Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Dataset Ultimate siap! Train: 1660, Val: 185


In [4]:
# --- 3. MEMANGGIL SANG MONSTER ---
print("Mendownload & Memuat ConvNeXt-V2 Base (Ini mungkin butuh waktu)...")
model = timm.create_model('convnextv2_base.fcmae_ft_in22k_in1k', pretrained=True, num_classes=len(class_names), drop_rate=0.4)
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

model_dir = '../models/'
os.makedirs(model_dir, exist_ok=True)
model_save_path = os.path.join(model_dir, 'convnextv2_base_ultimate.pth')

Mendownload & Memuat ConvNeXt-V2 Base (Ini mungkin butuh waktu)...


model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--timm--convnextv2_base.fcmae_ft_in22k_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [ ]:
# --- 4. TRAINING LOOP ---
best_val_acc = 0.0
patience = 3 
patience_counter = 0

print("MEMULAI PROSES TRAINING MONSTER...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for inputs, labels in train_pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * inputs.size(0)
        
    scheduler.step()
    epoch_train_loss = running_loss / len(train_dataset)
    
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]')
        for inputs, labels in val_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = correct / total
    
    print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), model_save_path)
        print(f"Monster terbaik disimpan! (Akurasi: {best_val_acc:.4f})")
        patience_counter = 0 
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print("Early Stopping aktif!")
        break

print(f"\nTraining Selesai! Model Ultimate siap dieksekusi dengan akurasi: {best_val_acc:.4f}")

🚀 MEMULAI PROSES TRAINING MONSTER...


Epoch 1/8 [Train]:   0%|          | 0/208 [00:00<?, ?it/s]

Epoch 1/8 [Val]:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.7038 | Val Loss: 0.3025 | Val Acc: 0.8811
Monster terbaik disimpan! (Akurasi: 0.8811)


Epoch 2/8 [Train]:   0%|          | 0/208 [00:00<?, ?it/s]

...

Epoch 7/8 [Val]:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 7 | Train Loss: 0.0098 | Val Loss: 0.1209 | Val Acc: 0.9514


Epoch 8/8 [Train]:   0%|          | 0/208 [00:00<?, ?it/s]

Epoch 8/8 [Val]:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 8 | Train Loss: 0.0057 | Val Loss: 0.1304 | Val Acc: 0.9676
Monster terbaik disimpan! (Akurasi: 0.9676)

Training Selesai! Model Ultimate siap dieksekusi dengan akurasi: 0.9676
